<a href="https://colab.research.google.com/github/hwanginseo04/-/blob/main/5%EC%9B%9429%EC%9D%BC%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn gradio sqlalchemy

In [3]:
import gradio as gr
from sqlalchemy import create_engine, Column, Integer, String, Boolean
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import sessionmaker, Session

# 데이터베이스 설정
SQLALCHEMY_DATABASE_URL = "sqlite:///./library.db"
engine = create_engine(SQLALCHEMY_DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()

class DBBook(Base):
    __tablename__ = "books"
    id = Column(Integer, primary_key=True, index=True)
    title = Column(String, index=True, nullable=False)
    author = Column(String, nullable=False)
    category = Column(String, nullable=False)
    is_borrowed = Column(Boolean, default=False)
    borrower = Column(String, nullable=True)

Base.metadata.create_all(bind=engine)

# ─────────────────────────────────────────────
# 🔥 [자동 데이터 삽입] 실행 시 책 3권 자동 등록
# ─────────────────────────────────────────────
def init_sample_data():
    db = SessionLocal()
    # 이미 데이터가 있으면 추가하지 않음
    if db.query(DBBook).first() is None:
        samples = [
            DBBook(id=1, title="파이썬 기초", author="홍길동", category="컴퓨터/IT"),
            DBBook(id=2, title="데이터 분석 입문", author="김철수", category="컴퓨터/IT"),
            DBBook(id=3, title="인공지능의 이해", author="이영희", category="인문학")
        ]
        db.add_all(samples)
        db.commit()
    db.close()

init_sample_data() # 실행하자마자 샘플 데이터 슝!

# 기능 함수들
def get_all_books_text():
    db = SessionLocal()
    books = db.query(DBBook).all()
    db.close()
    if not books: return "등록된 도서가 없습니다."
    output = "ID | 제목 | 저자 | 상태 | 대출자\n" + "-"*50 + "\n"
    for b in books:
        status = "❌ 대출중" if b.is_borrowed else "✅ 대출가능"
        output += f"{b.id} | {b.title} | {b.author} | {status} | {b.borrower or '-'}\n"
    return output

def process_borrow(book_id, borrower_name):
    db = SessionLocal()
    try:
        b = db.query(DBBook).filter(DBBook.id == int(book_id)).first()
        if not b: return "없는 책입니다.", get_all_books_text()
        if b.is_borrowed: return "이미 대출된 책입니다.", get_all_books_text()
        b.is_borrowed = True
        b.borrower = borrower_name
        db.commit()
        return f"'{b.title}' 대출 완료!", get_all_books_text()
    except: return "입력 오류", get_all_books_text()
    finally: db.close()

# 그라디오 화면
with gr.Blocks() as demo:
    gr.Markdown("# 📚 디지털 도서관 관리 프로그램")
    with gr.Row():
        with gr.Column():
            id_in = gr.Textbox(label="대상 도서 번호 (ID)")
            borrower_in = gr.Textbox(label="대출자 성함")
            btn_borrow = gr.Button("도서 대출하기", variant="primary")
            msg = gr.Textbox(label="결과 메시지")
        with gr.Column():
            board = gr.Textbox(value=get_all_books_text(), label="라이브러리 DB 전광판", lines=10)

    btn_borrow.click(process_borrow, [id_in, borrower_in], [msg, board])

# 실행
demo.launch(share=True)

/tmp/ipykernel_1783/3117501348.py:10: MovedIn20Warning: The ``declarative_base()`` function is now available as sqlalchemy.orm.declarative_base(). (deprecated since: 2.0) (Background on SQLAlchemy 2.0 at: https://sqlalche.me/e/b8d9)
  Base = declarative_base()


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1829d082b628f50197.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
